In [16]:
import pymupdf  # pip install pymupdf

In [5]:
import ollama
import pymupdf  # pip install pymupdf
import re

def summarize_pdf(pdf_path: str, model: str = "llama3.2") -> str:
    doc = pymupdf.open(pdf_path)
    text = "\n".join(page.get_text() for page in doc)
    
    # Chunk if too long (models have context limits)
    max_chars = 12000
    text = text[:max_chars]

    response = ollama.chat(
        model=model,
        messages=[{
            "role": "user",
            "content": f"Summarize this research paper concisely:\n\n{text}"
        }]
    )
    return response["message"]["content"]

In [89]:
def extract_section(pdf_path: str, start: str | list[str], end: str | list[str]) -> str | None:
    """
    Extract a section from a PDF using pymupdf text extraction, from a
    heading matching any of `start` up to a heading matching any of `end`.
    `start`/`end` accept a single keyword or a list of alternatives
    (e.g. start=["conclusion", "concluding remarks", "summary"]).
    Returns None if no matching start heading is found.
    """
    doc = pymupdf.open(pdf_path)
    text = "\n".join(page.get_text() for page in doc)

    def heading(keywords: str | list[str]) -> str:
        if isinstance(keywords, str):
            keywords = [keywords]
        alternation = "|".join(re.escape(k) for k in keywords)
        # A short line (up to 6 leading words) ending in one of the
        # keywords, e.g. "7\nDiscussion and Conclusion" or "Abstract".
        # \s+ matches the newline between a section number and its
        # title, so no separate numbering case is needed.
        return rf"^\s*(?:[\w.-]+\s+){{0,6}}(?:{alternation})\w*\s*$"

    pattern = re.compile(
        rf"(?im){heading(start)}\n(.*?)(?={heading(end)}|\Z)",
        re.DOTALL,
    )

    match = pattern.search(text)
    return match.group(1).strip() if match else None

In [92]:
extract_section('2607.pdf', 'conclusion', ['acknowledgements', 'references'])

'We develop an equity asset-pricing framework grounded in Kyle’s (1985) model of strategic informed\ntrading, in which the equilibrium price-impact coefficient λ governs how trading activity is impounded into\nprices. Using CRSP daily and monthly data for all equities in the CRSP universe from 2020 to 2025, we\nconstruct firm-month estimates of signed order flow and of λ itself, via both a direct within-month price-\nimpact regression and an Amihud-style illiquidity ratio. We then test their ability to forecast the cross-section\nof subsequent stock returns.\nThree findings emerge.\nFirst, signed order flow is a strong and highly significant predictor of both contemporaneous and one-\nmonth-ahead stock returns, robust to the inclusion of standard equity controls (size, book-to-market, mo-\nmentum, and Amihud illiquidity), consistent with Proposition 3’s prediction that order-flow innovations\nmove prices in equilibrium.\nSecond, volume volatility, our proxy for the noise-trading varian

In [ ]:
def pdf_to_text(pdf_path: str):
    doc = pymupdf.open(pdf_path)
    text = "\n".join(page.get_text() for page in doc)
    return text
    
result = pdf_to_text('2607.01550v1.pdf')

In [3]:
summarize_pdf('2606.pdf')

'Here is a concise summary of the research paper:\n\nThe study examines whether prediction markets match option prices, using Bitcoin as an example. The authors compare the prices of Polymarket Yes contracts with those of Binance call options for identical underlying assets (Bitcoin) and maturities (September 2023). They find a statistically significant positive wedge between the two prices, indicating that Polymarket Yes prices are consistently higher than Binance-implied risk-neutral values.\n\nThe study also finds that the gap between the two prices is persistent over intraday horizons with an autoregressive half-life of approximately four hours. This suggests that information and capital move across the two ecosystems but not instantaneously and not without cost.\n\nCross-sectional regressions reveal that the wedge is largest at low option-implied probabilities and long maturities, consistent with speculative demand for out-of-the-money prediction-market contracts.\n\nThe study con

# Scraping publication from the Arvix Website

Scrape Arvix and score author to determine crediblility

In [1]:
from scrape_credibility import get_score

In [2]:
get_score('Revisiting Trade-sign Long-memory and Square-root Law price impact')

35

In [8]:
import arxiv

ARVIX_FOUNDING_TIME = 19910101000000

def get_latest_microstructure_papers(n: int = 100, before_date=datetime.now()) -> list[dict]:
    """
    Fetch the latest `n` papers from arXiv q-fin.TR
    (Trading and Market Microstructure), returning title + authors.
    """
    before_date = before_date.strftime("%Y%m%d%H%M%S")
    client = arxiv.Client(page_size=n, delay_seconds=1)
    search = arxiv.Search(
        query=f"cat:q-fin.TR AND submittedDate:[{ARVIX_FOUNDING_TIME} TO {before_date}]",
        max_results=n,
        sort_by=arxiv.SortCriterion.SubmittedDate,
        sort_order=arxiv.SortOrder.Descending,
    )

    results = []
    for paper in client.results(search):
        results.append({
            "title": paper.title,
            "published": paper.published.date().isoformat(),
            "authors": [str(a) for a in paper.authors],
            "arxiv_id": paper.get_short_id(),
            "url": paper.entry_id,
        })

    return results


papers = get_latest_microstructure_papers(n=100)
print(f"Fetched {len(papers)} papers\n")
# for p in papers:
#     print(f"[{p['published']}] {p['title']}")
#     print(f"  Authors: {', '.join(p['authors'])}")
#     print(f"  {p['url']}\n")

Fetched 100 papers



In [1]:
from fetch_papers import get_latest_microstructure_papers, score_papers

In [16]:
papers[0]

{'title': 'Is Trend Still Your Friend?: A Microstructural Account of the Demise of Short-Term Trend-Following',
 'published': '2026-07-02',
 'authors': ['Jutta G. Kurth',
  'Zoltan Eisler',
  'Adam Rej',
  'Jean-Philippe Bouchaud'],
 'arxiv_id': '2607.01550v1',
 'url': 'http://arxiv.org/abs/2607.01550v1'}

In [2]:
papers = get_latest_microstructure_papers(n=100)
print(f"Fetched {len(papers)} papers\n")

Fetched 100 papers



In [27]:
papers, success, total = score_papers(papers)

[ERROR] OpenAlex works API returned 429
[ERROR] OpenAlex works API returned 429
[ERROR] OpenAlex works API returned 429
[ERROR] OpenAlex works API returned 429
[ERROR] OpenAlex works API returned 429
[ERROR] OpenAlex works API returned 429
[ERROR] OpenAlex works API returned 429
[ERROR] OpenAlex works API returned 429
[ERROR] OpenAlex works API returned 429
[ERROR] OpenAlex works API returned 429
[ERROR] OpenAlex works API returned 429
[ERROR] OpenAlex works API returned 429
[ERROR] OpenAlex works API returned 429
[ERROR] OpenAlex works API returned 429
[ERROR] OpenAlex works API returned 429
[ERROR] OpenAlex works API returned 429
[ERROR] OpenAlex works API returned 429
[ERROR] OpenAlex works API returned 429
[ERROR] OpenAlex works API returned 429
[ERROR] OpenAlex works API returned 429
[ERROR] OpenAlex works API returned 429
[ERROR] OpenAlex works API returned 429
[ERROR] OpenAlex works API returned 429
[ERROR] OpenAlex works API returned 429
[ERROR] OpenAlex works API returned 429


In [3]:

get_latest_microstructure_papers(n=100, before_date='2025-04-06')[0]

{'title': 'Information Leakages in the Green Bond Market',
 'published': '2025-04-04',
 'authors': ['Darren Shannon', 'Jin Gong', 'Barry Sheehan'],
 'arxiv_id': '2504.03311v1',
 'url': 'http://arxiv.org/abs/2504.03311v1'}

In [3]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

In [ ]:
MAX_WORKERS=10
def score_papers(papers: dict) -> tuple[list[dict], int, int]:
    success = 0
    total = 0
    paper_score = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as excecutor:
        title_async = {
            excecutor.submit(get_score, paper['title']): paper
            for paper in papers
        }

        for i, future in enumerate(as_completed(title_async), 1):
            paper = title_async[future]
            result = future.result()
            if isinstance(result, int):
                paper['Score'] = result
                paper_score.append(paper)
                success += 1
            total += 1
    return paper_score, success, total


In [24]:
papers, success, total = score_papers(papers)

# Determine relevance and Credibility

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Annotated
import operator
import manage_pdf as mp
from fetch_papers import find_and_score_papers
import io
import json

In [2]:
score_papers, success, total = find_and_score_papers(n=5)

In [7]:
for pap in score_papers:
    print(pap['Score'])

25
25
39


In [9]:
score_papers[0]

{'title': 'A Gabor--Epps uncertainty principle for traders',
 'published': '2026-07-05',
 'authors': ['Tim Gebbie'],
 'arxiv_id': '2607.04130v1',
 'url': 'http://arxiv.org/abs/2607.04130v1',
 'Score': 19}

In [3]:
class AI_Agent():
    def __init__(self, models : list[tuple[str, int]]) -> None:
        self.models = models 
        if len(self.models) == 0:
            self.index : int = - 1
        else:
            self.index : int = 0
            self.current_model = ChatGoogleGenerativeAI(
                model= self.models[self.index][0],
                temperature=0.7,
            )

    def get_model(self) -> str:
        if self.index >= len(self.models) or self.index < 0:
            return 'ERROR RAN OUT OF MODELS'
        return self.models[self.index][0]

    def next_index(self) -> int:
        if self.index >= len(self.models) - 1 or self.index < 0:
            self.index = -1
        else:
            self.index += 1
            self.current_model = ChatGoogleGenerativeAI(
                model= self.models[self.index][0],
                temperature=0.7,
            )

        return self.index

    def remaining_tokens(self) -> int:
        if self.index >= len(self.models) or self.index < 0:
            return -1
        return self.models[self.index][1]

    def update_model(self, prompt) -> None:
        tokens = self.current_model.get_num_tokens_from_messages(prompt)
        while tokens > self.models[self.index][1] and self.index != -1:
            self.next_index()
        if self.index == -1:
            raise Exception('Ran out of API tokens')
    def invoke(self, prompt) -> AIMessage:
        self.update_model(prompt)
        while self.index < len(self.models):
            try:
                response = self.current_model.invoke(prompt)
                return response
            except:
                self.next_index()
        raise Exception('Ran out of API calls')

    def stream(self, prompt):
        self.update_model(prompt)
        while self.index < len(self.models):
            try:
                response = self.current_model.stream(prompt)
                return response
            except:
                self.next_index()
        raise Exception('Ran out of API calls')

# for chunk in llm_with_search.stream(f"Summarize this text {result}"):
#     print(chunk.content, end='', flush=True)

In [4]:
ROLES = {
    'Relevance Filter' : {
        'system' : '''You are a researcher at a Quantitative finance firm and you want to determine if
the results of the current paper is relevant to the firm's research focus: {research_goal}

Evaluate relevance of the paper based on the following criteria:
- Does the paper address the market/asset/strategy outlined in the research's focus?
- THe paper must directly reference the research goals and must not indirectly relate to the firm's goal
- Does the Conclusion of the paper outline whether the hypothesis is valid and if the
  current strategy/method yield positive meaningful results not due to luck/overfitting
- If the Conclusion from the paper states that the methodology yielded no meaningful outcome, mark the paper's relevancy as false 
- Does the paper utilize data from recent times and does not over rely on overfitting
  to historical data

Respond with ONLY a single JSON object, using double-quoted keys/strings and no
markdown code fences, in exactly this format:
{{
    "relevancy": true or false,
    "reason": "1-2 sentences explaining why the paper is relevant",
    "confidence": "an integer from 1 to 3, with 1 being most confident and 3 being least confident"
}}

Do not include any text besides the JSON
''',
        'model' : 'llama3.2'
    },

    'Summarize Results': {
        'system': '''You are a researcher at a Quantitative finance firm and you want to Summarize the key findings
in the current paper.

When forming your summary, apply the following criterion:
- Identify and state the key findings from the paper
- Evaluate if the strategies in the paper have genuine merit or show signs of overfitting (Examples
overfitting model parameters, look ahead bias in price data, small out of sample testing).
- Explain the methodology the researcher's used and why they chose their method
- Note any criticisms you have regardings the methodology of the paper and the researcher's interpretation of their results

Respond with ONLY a single JSON object, using double-quoted keys/strings and no
markdown code fences, in exactly this format:
{{
    "Key Findings" : "2-4 sentences summarizing the key findings and if the results are valid or overfitted"
    "Methodology" : "2-4 sentences outlining the methods used and why"
    "criticisms" : "2-4 sentences describing weaknesses in the paper's methodology or results"
}}

Do not include any text besides the JSON
''',
        'model' : 'Gemini'
    }

}

class AgentState(TypedDict):
    messages: Annotated[list, operator.add]  # messages accumulate
    research_goal: str
    paper: dict
    paper_stream: io.BytesIO 
    agent: AI_Agent

relevance_schema = {
    "type": "object",
    "properties": {
        "relevancy": {"type": "boolean"},
        "reason": {"type": "string"},
        "confidence": {"type": "integer"},
    },
    "required": ["relevancy", "reason", "confidence"],
}

filter_llm = ChatOllama(
    model="llama3.2",
    temperature=0.7,
    format=relevance_schema,  # constrain Ollama to emit syntactically valid JSON
)

def _parse_json_response(content: str) -> dict:
    """Strip optional ```json fences before parsing an LLM JSON reply."""
    content = content.strip()
    if content.startswith("```"):
        content = re.sub(r"^```(?:json)?\s*|\s*```$", "", content).strip()
    return json.loads(content)

# ── Define a node (a step in the graph) ───────────────────────
def filter_papers(state: AgentState) -> AgentState:
    role = ROLES['Relevance Filter']
    filter_text = role['system'].format(research_goal=state['research_goal'])
    pdf_stream = state['paper_stream']
    abstract_section = mp.extract_section(pdf_stream,
                                          'abstract',
                                          'introduction')
    conclusion_section = mp.extract_section(pdf_stream,
                                            'conclusion',
                                            ['acknowledgements', 'references'])
    filter_prompt = [
        SystemMessage(content=filter_text),
        HumanMessage(f'''Here is the data to evaluate\n\n
                         Abstract: {abstract_section}\n\n
                         Conclusion: {conclusion_section}\n\n
                         ''')] 
    response = filter_llm.invoke(filter_prompt)
    return {"messages": [response]}


def should_continue(state: AgentState) -> str:
    last = state['messages'][-1]
    parsed = _parse_json_response(last.content)
    if parsed['relevancy'] == False:
        return END
    return "summarize_findings"

def summarize_findings(state: AgentState) -> AgentState:
    paper_str = mp.stream_to_text(state['paper_stream'])
    role = ROLES['Summarize Results']
    summarize_role = role['system']
    summarize_prompt = [
        SystemMessage(content=summarize_role),
        HumanMessage(f'''Here is the data to evaluate\n\n
                         {paper_str}''')
    ]
    response = state['agent'].invoke(summarize_prompt)
    return {"messages": [response]}

builder = StateGraph(AgentState)
builder.add_node("filter_llm", filter_papers)
builder.add_node("summarize_findings", summarize_findings)

builder.add_conditional_edges('filter_llm', should_continue)

builder.set_entry_point("filter_llm")

graph = builder.compile()

In [5]:
gemini_agents_list = [('gemini-2.5-flash', 250_000),
                  ('gemini-2.5-flash-lite', 250_000),
                  ('gemini-3-flash', 250_000),
                  ('gemini-3.1-flash-lite', 250_000),
                  ('gemini-3.5-flash', 250_000)
                  ]
Gemini_agents = AI_Agent(gemini_agents_list)

In [8]:
def wrap_by_words(text: str, words_per_line: int = 15) -> str:
    words = text.split(' ')
    lines = [
        " ".join(words[i:i + words_per_line])
        for i in range(0, len(words), words_per_line)
    ]
    return '\n'.join(lines)

In [13]:
paper = score_papers[paper_num]
paper_title = paper['title'] + '-' + paper['arxiv_id']

with open(f'saved_papers/{paper_title}.txt', 'w') as f:
    parsed = _parse_json_response(result["messages"][-1].content)
    key_findings = wrap_by_words('Key Findings:\n' + parsed['Key Findings']) + '\n\n'
    methods = wrap_by_words('Methods:\n' + parsed['Methodology']) + '\n\n'
    critiques = wrap_by_words('Critiques:\n' + parsed['criticisms']) + '\n\n'
    f.writelines([key_findings, methods, critiques])
    f.write(paper['url'])

In [18]:
score_papers[paper_num]['Score']

37

In [6]:
paper_num = 0

result = graph.invoke({
    'research_goal': 'The research goal is to find papers that provide unique insight for positive EV stock trading strategiesj',
    'paper': score_papers[paper_num],
    'paper_stream': mp.get_pdf_bytes(score_papers[paper_num]['arxiv_id']),
    'agent': Gemini_agents
})

print(result["messages"][-1].content)

{
    "Key Findings" : "The paper identifies periodic bursts in volatility and volume at quarter-hour marks in cryptocurrency futures, attributing them to algorithmic trading based on a sharp decline in trade-size roundness. The Autocorrelation Map reveals hidden serial dependence in order flow and returns at these boundaries. Quarter-hour opening returns are shown to be predictably out-of-sample (average R2 of 3.4%) using lagged returns and technical indicators, with order imbalance forecasting 4-to-12-hour returns. The robust out-of-sample testing and comprehensive analysis suggest these findings have genuine merit, rather than showing signs of overfitting.",
    "Methodology" : "Researchers used Binance perpetual contract trade data, aggregated to 10-second intervals. They applied a novel high-frequency trade-size roundness diagnostic to identify algorithmic activity. The Autocorrelation Map, a sign-based time-domain display, was introduced to visualize phase-specific dependence. Ou

In [13]:
json_test = json.loads(result['messages'][-1].content)

In [1]:
import yaml

In [2]:
import os
from dotenv import dotenv_values

# Bypass os.environ entirely — reads .env directly
config = dotenv_values()
# print(config)  # should show {'GOOGLE_API_KEY': '...'}

# Then set it manually
os.environ["GOOGLE_API_KEY"] = config["GOOGLE_API_KEY"]
os.environ["OPEN_ALEX_API_KEY"] = config["OPEN_ALEX_API_KEY"]

In [ ]:
import requests
import os
from dotenv import dotenv_values

config = dotenv_values()
api_key = config["GROQ_API_KEY"]
url = "https://api.groq.com/openai/v1/models"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers)

print(response.json())

In [18]:
result = mp.pdf_to_text('2607.01550v1.pdf')

In [19]:
llm.get_num_tokens(result)

27236

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Initialize Gemini model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7,
)

# llm_with_search = llm.bind(
#     tools=[{"google_search": {}}],
# )

# Ask a question
# for chunk in llm_with_search.stream(f"Summarize this text {result}"):
#     print(chunk.content, end='', flush=True)

In [1]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
import manage_pdf as mp
from fetch_papers import find_and_score_papers
from AI_Agent import AI_Agent
import io
import yaml
import json
import os
from dotenv import dotenv_values
import pickle
import re
from gitingest import ingest_async

In [2]:
with open('decision_tree.yaml', 'r') as f:
    data = yaml.safe_load(f)

In [31]:
print(f'{data["bins"]}')


[{'id': 'portfolio_simulation', 'title': 'Run a backtest / simulate a portfolio', 'triggers': ['backtest', 'simulate', 'from_signals', 'from_orders', 'buy/sell logic', 'position sizing', 'stop-loss', 'take-profit', 'fees/slippage'], 'files': [{'path': 'vectorbt/portfolio/base.py', 'role': 'entry_point', 'desc': "Portfolio class: .from_signals(), .from_orders(), .from_holding(), .from_random_signals(). Start here for any 'run a strategy' task."}, {'path': 'vectorbt/portfolio/nb.py', 'role': 'kernel', 'desc': 'Numba simulation loop - order execution logic, bar-by-bar state machine. Edit for custom fill/execution logic.'}, {'path': 'vectorbt/portfolio/enums.py', 'role': 'constants', 'desc': 'Order status, direction, size type enums - needed to interpret/construct order params.'}, {'path': 'vectorbt/portfolio/orders.py', 'role': 'results', 'desc': 'Orders records class - post-simulation order-level results.'}, {'path': 'vectorbt/portfolio/trades.py', 'role': 'results', 'desc': 'Trades/Posi

In [32]:
path = 'saved_papers/Signature-Based Optimal Execution for Statistical Arbitrage with Path-Dependent Trading Signals-2606.31387v1.yaml'

In [33]:
with open(path, 'r') as f:
    data2 = yaml.safe_load(f)

In [34]:
data2['Strategy Spec']

{'universe': 'SHEL and BP PLC equities on the London Stock Exchange (LSE)',
 'timeframe': 'Continuous (data frequency for signature features not explicitly specified, but z-score uses 8-hour rolling window)',
 'date_range': 'Jan 2025 - Dec 2025 (Training: Jan-Oct 2025, Test: Nov-Dec 2025)',
 'entry_rule': 'Trading speed `vt = Bxt` is continuously adjusted based on the current signature features `xt` and the optimized matrix `B`. The predictive signal `αt = Kxt` (derived from the z-score of the log spread) guides the direction of trading.',
 'exit_rule': 'No explicit discrete exit rule. Trading speed is continuously adjusted, and a terminal liquidation penalty `γ∥QT∥2` encourages unwinding positions by the end of the trading horizon.',
 'position_sizing': 'Position sizing is implicitly determined by the optimized trading speed `vt` (shares per unit time) and managed by penalties in the objective function, specifically for inventory risk (`ϕ Q⊤t ΣQt`) and dollar neutrality (`η(Q⊤t Pt)2`)

In [2]:
GITHUB_REPOS = ['https://github.com/polakowo/vectorbt']
RESEARCH_PATH = 'saved_papers/The Quarter-Hour Effect: Periodic Algorithmic Trading and Return Predictability in Cryptocurrency Futures-2607.09426v1.yaml'

In [25]:
bin_str = yaml.dump(data['bins'], sort_keys=False, default_flow_style=False)

In [13]:
# data = await fetch_github_repo(GITHUB_REPOS[0])
filter_llm = ChatOllama(
    model="qwen2.5-coder",
    temperature=0.7,
)
with open(RESEARCH_PATH, 'r') as f:
    txt_msg = f.read()
filter_prompt = [
    SystemMessage(content=txt_msg),
    HumanMessage('Using Python with vectorbt, generate code that backtests this strategy. This response will directly be run by python so ensure the full content will run without errors.')
]
response = filter_llm.invoke(filter_prompt)

2026-07-18 17:34:56.617 | INFO     | logging:callHandlers:1706 | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


In [14]:
with open('test.py', 'w') as f:
    f.write(response.content)

In [22]:
files = {}

chunks = re.split(r"={10,}\nFILE: (.+?)\n={10,}\n", content)

for path, body in zip(chunks[1::2], chunks[2::2]):
    files[path] = body

In [3]:
def file_chunks(file_str):
    files = {}

    chunks = re.split(r"={10,}\nFILE: (.+?)\n={10,}\n", file_str)

    for path, body in zip(chunks[1::2], chunks[2::2]):
        files[path] = body
    return files

async def fetch_github_repo(repo_title, exclude_pattern=[]):
    file_path = 'github_data.pkl'
    data = pickle.load(open(file_path, 'rb')) if os.path.exists(file_path) else {}
    if len(data.keys()) == 0 or repo_title not in data.keys():
        summary, tree, content = await ingest_async(
            repo_title,
            exclude_patterns=exclude_pattern,
        )
        files_data = file_chunks(content)
        data[repo_title] = {
                            'ingest': (summary, tree, content),
                            'files_data': files_data}
        pickle.dump(data, open(file_path, 'wb'))
    return data

# data = await fetch_github_repo("https://github.com/polakowo/vectorbt")

In [10]:
print(data['https://github.com/polakowo/vectorbt']['ingest'][1])

Directory structure:
└── polakowo-vectorbt/
    ├── README.md
    ├── conftest.py
    ├── Dockerfile
    ├── LICENSE.md
    ├── MANIFEST.in
    ├── mypy.ini
    ├── pyproject.toml
    ├── setup.py
    ├── .coveragerc
    ├── apps/
    │   └── candlestick-patterns/
    │       ├── README.md
    │       ├── app.py
    │       ├── Dockerfile
    │       ├── Procfile
    │       ├── requirements.txt
    │       └── assets/
    │           ├── default.css
    │           └── style.css
    ├── benchmarks/
    │   ├── README.md
    │   ├── bench_engine.py
    │   ├── bench_matrix.py
    │   ├── BENCHMARKS.md
    │   ├── BENCHMARKS_NUMBA.md
    │   └── BENCHMARKS_RUST.md
    ├── docs/
    │   ├── README.md
    │   ├── generate_api.py
    │   ├── LICENSE.md
    │   ├── mkdocs.yml
    │   ├── update_api_nav.py
    │   ├── docs/
    │   │   ├── CNAME
    │   │   ├── context7.json
    │   │   ├── index.md
    │   │   ├── assets/
    │   │   │   ├── interactive/
    │   │   │   │   ├── features_gbm

In [39]:
files_data = data['https://github.com/polakowo/vectorbt']['files_data']

In [41]:
print(files_data.keys())

dict_keys(['README.md', 'conftest.py', 'Dockerfile', 'LICENSE.md', 'MANIFEST.in', 'mypy.ini', 'pyproject.toml', 'setup.py', '.coveragerc', 'apps/candlestick-patterns/README.md', 'apps/candlestick-patterns/app.py', 'apps/candlestick-patterns/Dockerfile', 'apps/candlestick-patterns/Procfile', 'apps/candlestick-patterns/requirements.txt', 'apps/candlestick-patterns/assets/default.css', 'apps/candlestick-patterns/assets/style.css', 'benchmarks/README.md', 'benchmarks/bench_engine.py', 'benchmarks/bench_matrix.py', 'benchmarks/BENCHMARKS.md', 'benchmarks/BENCHMARKS_NUMBA.md', 'benchmarks/BENCHMARKS_RUST.md', 'examples/requirements-backtrader.txt', 'rust/README.md', 'rust/Cargo.toml', 'rust/pyproject.toml', 'rust/rustfmt.toml', 'rust/src/generic.rs', 'rust/src/indicators.rs', 'rust/src/labels.rs', 'rust/src/lib.rs', 'rust/src/portfolio.rs', 'rust/src/records.rs', 'rust/src/returns.rs', 'rust/src/signals.rs', 'tests/__init__.py', 'tests/conftest.py', 'tests/test_base.py', 'tests/test_data.py'

In [42]:
print(tree)

Directory structure:
└── polakowo-vectorbt/
    ├── README.md
    ├── conftest.py
    ├── Dockerfile
    ├── LICENSE.md
    ├── MANIFEST.in
    ├── mypy.ini
    ├── pyproject.toml
    ├── setup.py
    ├── .coveragerc
    ├── apps/
    │   └── candlestick-patterns/
    │       ├── README.md
    │       ├── app.py
    │       ├── Dockerfile
    │       ├── Procfile
    │       ├── requirements.txt
    │       └── assets/
    │           ├── default.css
    │           └── style.css
    ├── benchmarks/
    │   ├── README.md
    │   ├── bench_engine.py
    │   ├── bench_matrix.py
    │   ├── BENCHMARKS.md
    │   ├── BENCHMARKS_NUMBA.md
    │   └── BENCHMARKS_RUST.md
    ├── examples/
    │   └── requirements-backtrader.txt
    ├── rust/
    │   ├── README.md
    │   ├── Cargo.toml
    │   ├── pyproject.toml
    │   ├── rustfmt.toml
    │   └── src/
    │       ├── generic.rs
    │       ├── indicators.rs
    │       ├── labels.rs
    │       ├── lib.rs
    │       ├── portfolio.rs
    │  

In [57]:
import subprocess
import os
current_env = os.environ.copy()
process = subprocess.Popen(
        ['/bin/bash'],
        stdin=subprocess.PIPE, 
        stdout=subprocess.PIPE, 
        stderr=subprocess.PIPE, 
        text=True,
        env=current_env
    )

In [58]:
process.stdin.write('echo Hello')

10

In [ ]:
process.stdin.write('\n')  # terminate any pending partial command
process.stdin.write('python3 test.py 2>&1\n')
process.stdin.write('echo DONE\n')
process.stdin.flush()

lines = []
while True:
    line = process.stdout.readline()
    if not line or line.strip() == 'DONE':
        break
    print(line, end='')
    lines.append(line)

> [!NOTE]
> Run this command to initialize your project:
> ```bash
> npx create-react-app my-app
> ```